# NLP Lab Task 04
**Name / Reg No:** FA23-BAI-024

Run the first two cells first (they install/download what the lab needs: NLTK data and the spaCy model). Internet access is needed once.

In [ ]:
import sys, subprocess
def _pip(*pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try: import nltk
except ImportError: _pip("nltk"); import nltk
try: import spacy
except ImportError: _pip("spacy"); import spacy
for pkg in ["stopwords", "movie_reviews", "brown"]:
    nltk.download(pkg, quiet=True)
try: nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

In [ ]:
import re, html, time, itertools, random
from collections import Counter, defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 200)

## Task 1: Lowercasing Basics and Its Drawbacks

In [ ]:
# 1. Corpus of 12 mixed-case sentences
corpus = [
    "Apple released a new iPhone in the US yesterday.",
    "I ate an apple and a banana for breakfast.",
    "APPLE stock jumped after the announcement.",
    "The US government told us to stay calm.",
    "Please give us a few minutes to finish.",
    "Mark my words, Mark will win the race.",
    "We visit Turkey every summer and eat turkey on holiday.",
    "I may visit Paris in May.",
    "Bill paid the bill on time.",
    "Polish people often polish their furniture.",
    "He works at Amazon, but the amazon river is huge.",
    "WOW, that was GREAT!",
]
tok = lambda s: re.findall(r"\b\w+\b", s)

# 2. Vocabulary before / after lowercasing
vocab_before = {w for s in corpus for w in tok(s)}
vocab_after  = {w.lower() for s in corpus for w in tok(s)}
print("Vocabulary before lowercasing:", len(vocab_before))
print("Vocabulary after  lowercasing:", len(vocab_after))
print("Reduction:", len(vocab_before) - len(vocab_after), "word types merged")

# 3. Words whose different casings collapse into one form
groups = defaultdict(set)
for w in vocab_before: groups[w.lower()].add(w)
print("\nForms merged by lowercasing:")
for low, forms in sorted(groups.items()):
    if len(forms) > 1: print(f"  {sorted(forms)} -> '{low}'")

In [ ]:
amb = pd.DataFrame([
    ("US / us",       "The US government told us to stay calm.",        "'us' is ambiguous: country (proper noun) vs. pronoun"),
    ("Apple / apple", "Apple released a new iPhone... / I ate an apple", "'apple': company vs. fruit"),
    ("Turkey / turkey","We visit Turkey ... and eat turkey",             "'turkey': country vs. bird"),
    ("Polish / polish","Polish people often polish their furniture",     "'polish': nationality/language vs. verb"),
], columns=["Word", "Example", "Ambiguity after lowercasing"])
display(amb)

**4. Which NLP tasks can lowercasing harm?**

- **Named Entity Recognition (NER):** capitalisation is one of the strongest cues for proper nouns. After lowercasing, "Apple" (company), "US" (country) and "Turkey" (country) look like ordinary words, so the model loses a key feature and confuses entity types.
- **Sentiment analysis:** emphasis such as "GREAT!!!" or "TERRIBLE" signals intensity. Lowercasing removes that signal, so strongly and mildly positive/negative texts look the same. Words like "US"/"us" can also be misread.
- **Search / information retrieval:** a query for "US" (country) or "Apple" (company) will also match "us" (pronoun) and "apple" (fruit), which lowers precision. Acronyms (e.g. "IT" vs. "it", "WHO" vs. "who") suffer in the same way.
- Lowercasing is still useful for reducing sparsity (smaller vocabulary), so it should be applied only when the task does not depend on case (e.g. topic classification).

## Task 2: Noise Removal Pipeline

In [ ]:
# 1. Noisy dataset (16 tweets / reviews)
noisy = [
    "Check this out!!! https://t.co/abc123 @john_doe #Awesome 😍😍",
    "<p>Great product!</p> <br> Works with C++ and Python   perfectly 👍",
    "@amazon my order is late again... www.amazon.com/help #fail 😡",
    "<div>Only $100 for this laptop!!!</div> Best price ever #deal",
    "Learning C++ and C# this semester   #coding http://bit.ly/xyz",
    "RT @newsbot: Breaking: markets fall 5% today &amp; investors panic 😱 https://news.com/a?b=1",
    "<b>LOVED</b> it!!! 10/10 would buy again 😊😊 #happy",
    "Terrible service...   waited 2 hours @support  #neveragain",
    "Visit https://example.com/page?id=5&x=2 for 50% off!!!",
    "<a href='http://spam.com'>Click here</a> to win $1,000 NOW",
    "Wow   so    many   spaces    here!!",
    "@alice @bob thanks for the help :) #teamwork",
    "The battery lasts 12 hrs; screen is 15.6\" — nice! 👌",
    "I paid $20 for this &amp; it broke in 2 days 😤 #waste",
    "<h1>Review</h1><p>Node.js &gt; PHP, in my opinion.</p> #webdev",
    "email me at help@shop.com or call 555-1234 !!!",
]
print(len(noisy), "entries")

# 2. One regex function per noise type
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF\uFE0F\u200d]+", flags=re.UNICODE)
def remove_urls(t):     return re.sub(r"(https?://|www\.)\S+", "", t)
def remove_html(t):     return html.unescape(re.sub(r"<[^>]+>", " ", t))
def remove_emojis(t):   return EMOJI.sub("", t)
def handle_social(t):
    t = re.sub(r"@\w+", "", t)            # drop @mentions
    return re.sub(r"#(\w+)", r"\1", t)    # keep hashtag word, drop '#'
def remove_punct(t):    return re.sub(r"[^\w\s]", "", t)
def normalize_ws(t):    return re.sub(r"\s+", " ", t).strip()

STEPS = {"html": remove_html, "urls": remove_urls, "emoji": remove_emojis,
         "social": handle_social, "punct": remove_punct, "ws": normalize_ws}
DEFAULT_ORDER = ["html", "urls", "emoji", "social", "punct", "ws"]

# 3. Combined pipeline
def clean_text(text, order=DEFAULT_ORDER, steps=STEPS):
    for name in order:
        text = steps[name](text)
    return text

for t in noisy[:3]:
    print(repr(t)); print("  ->", repr(clean_text(t)))

In [ ]:
# Effect of ordering: try all 720 permutations of the six steps
default_out = [clean_text(t) for t in noisy]
variants = {}
for perm in itertools.permutations(DEFAULT_ORDER):
    out = tuple(clean_text(t, order=perm) for t in noisy)
    variants.setdefault(out, []).append(perm)
print("Distinct outputs over 720 orderings:", len(variants))
same = sum(1 for p in itertools.permutations(DEFAULT_ORDER) if [clean_text(t, p) for t in noisy] == default_out)
print("Orderings giving the same output as the default:", same)

# concrete examples
bad_orders = {
    "punctuation BEFORE urls":   ["punct", "urls", "html", "emoji", "social", "ws"],
    "whitespace BEFORE others":  ["ws", "html", "urls", "emoji", "social", "punct"],
    "punctuation BEFORE social": ["punct", "social", "html", "urls", "emoji", "ws"],
}
sample = noisy[0]; sample2 = noisy[2]; sample3 = noisy[10]
print("\nDefault          :", repr(clean_text(sample)))
print("punct before urls:", repr(clean_text(sample, bad_orders["punctuation BEFORE urls"])))
print("Default          :", repr(clean_text(sample2)))
print("punct before soc.:", repr(clean_text(sample2, bad_orders["punctuation BEFORE social"])))
print("ws first         :", repr(clean_text("<b>a</b>  b", bad_orders["whitespace BEFORE others"])), "(extra spaces left by removed tags)")

**Does the order matter? Yes.** If punctuation is removed *before* URLs, `https://t.co/abc123` becomes `httpstcoabc123` and the URL regex can no longer recognise it, so junk tokens stay. If punctuation is removed before hashtags/mentions, `@` and `#` disappear and the mention/hashtag rules do nothing. If whitespace is normalised too early, removing tags or emojis later leaves new double spaces. A safe order is: **HTML → URLs → emojis → mentions/hashtags → punctuation → whitespace**.

In [ ]:
# 4. Before / after table and over-cleaning
idx = [1, 3, 4, 8, 12]
tbl = pd.DataFrame({"Before": [noisy[i] for i in idx], "After (clean_text)": [clean_text(noisy[i]) for i in idx]})
display(tbl)

**Over-cleaning:** the plain `remove_punct` destroys meaningful symbols: `C++` → `C`, `C#` → `C`, `$100` → `100` (currency lost), `50%` → `50`, `15.6"` → `156` (decimal point removed, changing the number!), `Node.js` → `Nodejs`. Below is a safer version that protects these patterns.

In [ ]:
PROTECT = re.compile(r"[A-Za-z]\+\+|[A-Za-z]#|\$\d+(?:[.,]\d+)*|\d+(?:\.\d+)?%|\d+\.\d+")
def remove_punct_safe(t):
    out, last = [], 0
    for m in PROTECT.finditer(t):
        out.append(re.sub(r"[^\w\s]", "", t[last:m.start()])); out.append(m.group()); last = m.end()
    out.append(re.sub(r"[^\w\s]", "", t[last:]))
    return "".join(out)

SAFE_STEPS = {**STEPS, "punct": remove_punct_safe}
cmp_ = pd.DataFrame({"Before": [noisy[i] for i in idx],
                     "Plain clean": [clean_text(noisy[i]) for i in idx],
                     "Safe clean": [clean_text(noisy[i], steps=SAFE_STEPS) for i in idx]})
display(cmp_)

## Task 3: Stop-Word Removal and Its Effect on Vocabulary

In [ ]:
from nltk.corpus import stopwords, movie_reviews

# 1. NLTK stop-word list
sw_list = stopwords.words("english")
sw = set(sw_list)
print("First 20:", sw_list[:20])
print("Total size:", len(sw_list))

# 2. Dataset: 300 random movie reviews, kept as sentences
fids = random.Random(0).sample(movie_reviews.fileids(), 300)
def prep(sent): return [w.lower() for w in sent if w.isalpha() or w == "n't"]
sents_before = [s for s in (prep(x) for f in fids for x in movie_reviews.sents(f)) if s]
sents_after  = [s for s in ([w for w in sent if w not in sw] for sent in sents_before) if s]

def stats(sents):
    toks = [w for s in sents for w in s]
    return len(toks), len(set(toks)), len(toks) / len(sents)
b, a = stats(sents_before), stats(sents_after)
display(pd.DataFrame({"Before": b, "After": a}, index=["Token count", "Vocabulary size", "Avg sentence length"]).round(2))

In [ ]:
# 3. Top-15 words before / after
def top15(sents): return Counter(w for s in sents for w in s).most_common(15)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, sents, title in zip(axes, [sents_before, sents_after], ["Before stop-word removal", "After stop-word removal"]):
    words, counts = zip(*top15(sents))
    ax.barh(words[::-1], counts[::-1]); ax.set_title(title); ax.set_xlabel("Frequency")
plt.tight_layout(); plt.show()

In [ ]:
# 4. Custom stop-word list for movie reviews
ADD = {
    "movie": "Appears in nearly every review; carries no sentiment or topic information in this domain.",
    "movies": "Plural of 'movie', same reason.",
    "film": "Domain word present in almost all reviews; non-discriminative.",
    "films": "Plural of 'film', same reason.",
    "one": "Very frequent filler ('one of the best', 'one thing') with little meaning.",
    "also": "Discourse filler; does not help distinguish positive from negative.",
    "even": "Very frequent adverb with little standalone meaning.",
    "would": "Frequent modal verb; adds noise.",
    "could": "Frequent modal verb; adds noise.",
    "get": "High-frequency generic verb.",
    "make": "High-frequency generic verb.",
}
KEEP = {
    "not": "Negation: 'not good' vs 'good' have opposite sentiment.",
    "no": "Negation ('no plot', 'no problem').",
    "nor": "Negation conjunction.",
    "n't": "Contraction of 'not' (tokenised as n't); removing it flips sentiment.",
    "very": "Intensifier: 'very bad' is stronger than 'bad'.",
    "too": "Intensifier / excess ('too long', 'too slow').",
    "against": "Carries opposition meaning ('against the odds').",
    "but": "Contrast marker: 'good acting but weak story'.",
    "only": "Restricts meaning ('only good thing').",
}
custom_sw = (sw - set(KEEP)) | set(ADD)
print("Added:", len(ADD), "words | Removed (kept in text):", len(KEEP), "words | New list size:", len(custom_sw))
display(pd.DataFrame([(w, "ADD to stop list", r) for w, r in ADD.items()] + [(w, "REMOVE from stop list (keep)", r) for w, r in KEEP.items()],
                     columns=["Word", "Action", "Justification"]))
sents_custom = [s for s in ([w for w in sent if w not in custom_sw] for sent in sents_before) if s]
display(pd.DataFrame({"Default NLTK": stats(sents_after), "Custom list": stats(sents_custom)},
                     index=["Token count", "Vocabulary size", "Avg sentence length"]).round(2))

## Task 4: Risks of Stop-Word Removal (Sentiment, MT, Chatbots)

In [ ]:
tokenize_words = lambda s: re.findall(r"[a-z]+(?:'[a-z]+)?", s.lower())
remove_sw = lambda toks, stop=sw: [t for t in toks if t not in stop]

POS = {"good","great","happy","like","love","excellent","nice","helpful","easy"}
NEG = {"bad","poor","terrible","hate","problem","slow","broken","awful"}
NEGATORS = {"not","no","never","nor","neither","hardly","isn't","wasn't","don't","didn't","doesn't","can't","won't","aren't","weren't"}
def sentiment(tokens):
    score = 0
    for i, t in enumerate(tokens):
        v = 1 if t in POS else -1 if t in NEG else 0
        if v and any(p in NEGATORS for p in tokens[max(0, i-3):i]): v = -v   # negation flips next sentiment word
        score += v
    return score
label = lambda s: "positive" if s > 0 else "negative" if s < 0 else "neutral"

# 1. Negation sentences
neg_sents = [
    "This phone is not good.",
    "I am never happy with this service.",
    "There is no problem with the delivery.",
    "I don't like the food here.",
    "The service wasn't bad at all.",
]
rows = []
for s in neg_sents:
    t = tokenize_words(s); r = remove_sw(t)
    sb, sa = sentiment(t), sentiment(r)
    rows.append((s, " ".join(r), label(sb), label(sa), "YES" if label(sb) != label(sa) else "no"))
display(pd.DataFrame(rows, columns=["Sentence", "After stop-word removal", "Sentiment before", "Sentiment after", "Flipped?"]))

In [ ]:
# 2. Machine translation: sentences and their stop-word-reduced versions
mt_sents = ["I do not want to go to the market today.",
            "She is not at home because the doctor told her to rest.",
            "The book on the table is mine."]
mt = pd.DataFrame({"Original": mt_sents, "Reduced": [" ".join(remove_sw(tokenize_words(s))) for s in mt_sents]})
display(mt)

**Urdu translation:** I cannot call Google Translate from this notebook, so paste the real output of the translator into the table below. The Urdu lines already there are my own hand translations, written to show the expected effect; **replace them with Google Translate's output** before submitting.

In [ ]:
mt_urdu = pd.DataFrame({
    "Original (EN)":  mt_sents,
    "Urdu (original)": ["میں آج بازار نہیں جانا چاہتا۔",
                        "وہ گھر پر نہیں ہے کیونکہ ڈاکٹر نے اسے آرام کرنے کو کہا ہے۔",
                        "میز پر رکھی کتاب میری ہے۔"],
    "Reduced (EN)":   mt["Reduced"],
    "Urdu (reduced)":  ["چاہتا جانا بازار آج",
                        "گھر ڈاکٹر کہا آرام",
                        "کتاب میز میری"],
})
display(mt_urdu)

**Observation:** the full sentences translate into fluent Urdu with the negation (`نہیں`) and relations (`پر`, `کیونکہ`) intact. The reduced versions lose the negation in sentences 1 and 2, so the translation **says the opposite** ("wants to go to the market", "is at home"), and lose grammar words so the Urdu becomes a broken list of words. Stop-words carry negation, tense and relations that MT needs, so they should never be removed before translation.

In [ ]:
# 3. Chatbot intent
queries = ["How to not be charged?",
           "What is the status of my order?",
           "Why can't I log in to my account?",
           "I do not want to cancel my subscription.",
           "How can I get a refund for this item?"]
rows = []
for q in queries:
    t = tokenize_words(q); r = remove_sw(t)
    rows.append((q, " ".join(r), ", ".join(w for w in t if w in sw)))
display(pd.DataFrame(rows, columns=["Query", "After stop-word removal", "Words lost"]))

**Intent damage:** "How to not be charged?" → `charged` (the user wants to *avoid* charges, but the reduced text looks like a billing complaint). "I do not want to cancel my subscription" → `want cancel subscription` (a *retention* intent becomes a *cancellation* intent). "Why can't I log in" → `log account` (the question type and the failure are lost). "What is the status of my order?" → `status order` is still understandable, but the question word is lost, which can matter for slot filling. Negations and question words (how/what/why) are the intent signals.

In [ ]:
# 4. Safe strategy: keep negations, question words, modals, intensifiers/contrast words
KEEP_WORDS = {"not","no","nor","never","but","very","too","against","only","off",
              "how","what","why","when","where","who","which","can","will","should"}
NEG_FRAGMENTS = {"didn","doesn","don","isn","wasn","weren","aren","couldn","shouldn","wouldn",
                 "hasn","haven","hadn","mustn","needn","shan","won","mightn","ain"}
safe_sw = {w for w in sw if not (w.endswith("n't") or w in KEEP_WORDS or w in NEG_FRAGMENTS)}
print("Default stop-words:", len(sw), "| Safe stop-words:", len(safe_sw))

rows = []
for s in neg_sents:
    t = tokenize_words(s); r_def, r_safe = remove_sw(t), remove_sw(t, safe_sw)
    sb = label(sentiment(t))
    rows.append((s, " ".join(r_def), label(sentiment(r_def)), " ".join(r_safe), label(sentiment(r_safe)), sb))
res = pd.DataFrame(rows, columns=["Sentence", "Default removal", "Default label", "Safe removal", "Safe label", "True (original)"])
display(res)
print("Flipped with default list:", (res["Default label"] != res["True (original)"]).sum(), "/ 5")
print("Flipped with safe list   :", (res["Safe label"]    != res["True (original)"]).sum(), "/ 5")
for q in queries[:2] + queries[3:4]:
    print(q, "->", " ".join(remove_sw(tokenize_words(q), safe_sw)))

## Task 5: Hands-On with the Porter Stemmer

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer
porter, snowball = PorterStemmer(), SnowballStemmer("english")

# 1. 46 words, each with the root (lemma) a linguist would expect
words = {
 "running":"run","runs":"run","ran":"run","studies":"study","studying":"study","studied":"study",
 "happily":"happy","happiness":"happy","generalization":"general","generalize":"general","general":"general",
 "generous":"generous","generously":"generous","better":"good","best":"good","was":"be","were":"be","is":"be","are":"be",
 "cats":"cat","mice":"mouse","mouse":"mouse","university":"university","universe":"universe","universal":"universal",
 "organization":"organize","organ":"organ","news":"news","new":"new","flies":"fly","flying":"fly","fly":"fly",
 "caring":"care","cares":"care","cared":"care","operate":"operate","operation":"operate","operating":"operate",
 "connection":"connect","connected":"connect","connecting":"connect","fairly":"fair","data":"data","datum":"datum",
}
# 2. Table: Correct if the stem equals the stem of the true root (i.e. related forms collapse together)
rows = [(w, porter.stem(w), "Correct" if porter.stem(w) == porter.stem(root) else "Incorrect", root) for w, root in words.items()]
tab = pd.DataFrame(rows, columns=["Word", "Stem", "Correct/Incorrect", "Expected root"])
display(tab)
print(tab["Correct/Incorrect"].value_counts().to_dict())

In [ ]:
# 3. Over-stemming: one stem shared by words with DIFFERENT roots
by_stem = defaultdict(list)
for w, root in words.items(): by_stem[porter.stem(w)].append((w, root))
print("OVER-STEMMING (different words, same stem):")
for stem, items in by_stem.items():
    if len({r for _, r in items}) > 1: print(f"  stem '{stem}': {[w for w, _ in items]}")

# Under-stemming: ONE root whose forms get DIFFERENT stems
by_root = defaultdict(set)
for w, root in words.items(): by_root[root].add((w, porter.stem(w)))
print("\nUNDER-STEMMING (related words, different stems):")
for root, items in by_root.items():
    if len({s for _, s in items}) > 1: print(f"  root '{root}': {sorted(items)}")

In [ ]:
# 4. Porter vs Snowball
extra = ["abilities","dying","lying","skies","sky","early","only","ugly","gently","proceed","exceed","succeed","herring","outing","fairly","generously"]
cands = list(dict.fromkeys(list(words) + extra))
diff = [(w, porter.stem(w), snowball.stem(w)) for w in cands if porter.stem(w) != snowball.stem(w)]
print(f"{len(diff)} of {len(cands)} words differ")
display(pd.DataFrame(diff, columns=["Word", "Porter", "Snowball"]))

**Notes:** Porter is a rule-based suffix stripper, so it over-stems when different words share a prefix (`university`/`universe`/`universal` → `univers`, `general`/`generous` → `gener`, `organ`/`organization` → `organ`, `news`/`new` → `new`) and under-stems irregular forms (`ran`/`run`, `mice`/`mouse`, `better`/`good`, `was`/`be`). Snowball (Porter2) fixes some Porter quirks, e.g. adverbs such as `fairly`, so the two stemmers disagree on a number of words.

## Task 6: Lemmatization (spaCy) vs. Stemming

In [ ]:
para = ("The researchers were studying how children learn languages better when they are running around outside. "
        "She was happier after the doctors said her studies had improved, and the teachers noticed that the students "
        "were reading more books than before. Last year, the mice that were living in the old building ate the "
        "papers, so the staff bought better boxes and carefully moved the leaves, the wolves' photographs and the "
        "families' letters to a safer room. They had been working for hours, and although everyone felt tired, the "
        "organizers kept talking about the best ways of connecting people through generous donations and caring volunteers.")
print("Paragraph length:", len(para.split()), "words")

# 1. spaCy tokens, lemmas, POS tags
doc = nlp(para)
toks = [t for t in doc if not t.is_punct and not t.is_space]

# 2. Comparison table
stemmer = PorterStemmer()
cmp6 = pd.DataFrame([(t.text, t.pos_, stemmer.stem(t.text), t.lemma_) for t in toks], columns=["Token", "POS", "Stem", "Lemma"])
display(cmp6.head(40))

In [ ]:
# 3. Where lemmatization does better than stemming
better = cmp6[(cmp6["Stem"].str.lower() != cmp6["Lemma"].str.lower()) & (cmp6["Token"].str.lower() != cmp6["Lemma"].str.lower())].drop_duplicates("Token")
print("Words where the lemma is a real dictionary word and the stem is not / differs:")
display(better)

# role of POS tag
for s in ["She sings better than him.", "I saw the old saw.", "The leaves are green; he leaves tomorrow."]:
    print(s, "->", [(t.text, t.pos_, t.lemma_) for t in nlp(s) if t.text.lower() in ("better", "saw", "leaves")])

**Role of the POS tag:** a lemmatizer needs the part of speech to pick the right dictionary form. The same surface form can have different lemmas depending on POS (`saw` as a verb → `see`, as a noun → `saw`; `leaves` as a noun → `leaf`, as a verb → `leave`; `better` as an adjective → `good`). A stemmer ignores context, so it applies the same suffix rules to every occurrence and also cannot handle irregular forms like `was → be` or `mice → mouse`.

In [ ]:
# 4. Speed comparison
big = para * 50
words_big = re.findall(r"[A-Za-z']+", big)
t0 = time.perf_counter(); _ = [stemmer.stem(w) for w in words_big]; t_stem = time.perf_counter() - t0
t0 = time.perf_counter(); _ = [t.lemma_ for t in nlp(big)]; t_lemma = time.perf_counter() - t0
print(f"Words processed        : {len(words_big)}")
print(f"Porter stemming        : {t_stem:.4f} s")
print(f"spaCy lemmatization    : {t_lemma:.4f} s  (includes tokenizing + POS tagging)")
print(f"Lemmatization is {t_lemma / t_stem:.1f}x slower")

**When to use which:**
- **Stemming** when speed and memory matter and exact words don't: large-scale search/indexing (e.g. Elasticsearch), quick text classification baselines, very large corpora.
- **Lemmatization** when meaning and readable output matter: chatbots, question answering, information extraction, topic modelling where the output words are shown to people, and any task where irregular forms matter. It is slower because it needs POS tagging.

## Task 7: Preprocessing Pipeline

In [ ]:
class Preprocessor:
    """lowercase -> noise removal -> tokenize -> stop-words -> stem OR lemmatize (each step optional)."""
    def __init__(self, lowercase=True, noise=True, stop=False, stem=False, lemma=False, stop_set=None):
        assert not (stem and lemma), "choose stemming OR lemmatization"
        self.lowercase, self.noise, self.stop, self.stem, self.lemma = lowercase, noise, stop, stem, lemma
        self.stop_set = stop_set or set(stopwords.words("english"))
        self.stemmer = PorterStemmer()
    def _prep(self, text):
        if self.lowercase: text = text.lower()
        if self.noise: text = clean_text(text, order=["html", "urls", "emoji", "social", "ws"])
        return text
    def transform(self, docs):
        texts = [self._prep(d) for d in docs]
        out = []
        if self.lemma:                               # spaCy does tokenizing + lemmatizing
            for d in nlp.pipe(texts, batch_size=32, disable=["parser", "ner"]):
                toks = [t for t in d if t.is_alpha]
                if self.stop: toks = [t for t in toks if t.text.lower() not in self.stop_set]
                out.append([t.lemma_.lower() for t in toks])
        else:
            for text in texts:
                toks = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text)
                if self.stop: toks = [t for t in toks if t.lower() not in self.stop_set]
                if self.stem: toks = [self.stemmer.stem(t) for t in toks]
                out.append(toks)
        return out

# dataset: 600 movie reviews (balanced), labels 1 = positive
rng = random.Random(1)
neg_ids = rng.sample(movie_reviews.fileids("neg"), 300); pos_ids = rng.sample(movie_reviews.fileids("pos"), 300)
docs = [movie_reviews.raw(f) for f in neg_ids + pos_ids]
y = np.array([0] * 300 + [1] * 300)
print(len(docs), "documents")

In [ ]:
# 2. Four configurations
configs = {
    "1. lowercase only":           dict(stop=False, stem=False, lemma=False),
    "2. lowercase + stop-words":   dict(stop=True,  stem=False, lemma=False),
    "3. lowercase + stemming":     dict(stop=False, stem=True,  lemma=False),
    "4. lowercase + lemmatization":dict(stop=False, stem=False, lemma=True),
}
processed = {}
for name, kw in configs.items():
    t0 = time.perf_counter()
    processed[name] = Preprocessor(lowercase=True, noise=True, **kw).transform(docs)
    print(f"{name:<32} done in {time.perf_counter() - t0:5.1f}s")

# 3. Vocabulary and avg tokens per document
summary = pd.DataFrame([(n, len({t for d in p for t in d}), np.mean([len(d) for d in p])) for n, p in processed.items()],
                       columns=["Configuration", "Vocabulary size", "Avg tokens / doc"]).round(1)
display(summary)

In [ ]:
# 4. Classification: TF-IDF + Naive Bayes / Logistic Regression, 5-fold cross-validation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

def identity(x): return x
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, p in processed.items():
    row = [name]
    for model in (MultinomialNB(), LogisticRegression(max_iter=1000)):
        pipe = Pipeline([("tfidf", TfidfVectorizer(tokenizer=identity, preprocessor=identity, token_pattern=None, lowercase=False)),
                         ("clf", model)])
        row.append(cross_val_score(pipe, p, y, cv=cv, scoring="accuracy").mean())
    rows.append(row)
acc = pd.DataFrame(rows, columns=["Configuration", "Naive Bayes acc.", "Logistic Reg. acc."])
acc["Mean"] = acc[["Naive Bayes acc.", "Logistic Reg. acc."]].mean(axis=1)
display(acc.round(4))

best = acc.loc[acc["Mean"].idxmax()]; worst = acc.loc[acc["Mean"].idxmin()]
print("Best configuration :", best["Configuration"], f"(mean acc {best['Mean']:.4f})")
print("Worst configuration:", worst["Configuration"], f"(mean acc {worst['Mean']:.4f})")
print(f"Spread between best and worst: {(best['Mean'] - worst['Mean']) * 100:.2f} percentage points")

**Analysis (5-6 lines; read it together with the numbers printed above):**
1. The best configuration is the one printed above; the gap between the best and worst configuration is usually small (a few percentage points), because TF-IDF on 600 reviews is already dominated by strong sentiment words.
2. Stemming and lemmatization shrink the vocabulary and merge related forms (`bore`, `boring`, `bored`), which reduces sparsity and often helps small datasets.
3. Plain stop-word removal can hurt or give no gain for sentiment: it deletes negations such as `not` and `n't` (see Task 4) and TF-IDF already down-weights common words.
4. Lemmatization keeps real words and handles irregular forms but is much slower than stemming for a similar accuracy.
5. With only 600 documents and 5-fold CV the differences are within noise, so choose the cheapest configuration that is not clearly worse, and verify on a larger dataset.

## Task 8: Implementing Minimum Edit Distance (Levenshtein)

In [ ]:
# 1. Recursive Levenshtein (no memoization)
calls = 0
def lev_recursive(a, b):
    global calls
    calls += 1
    if not a: return len(b)
    if not b: return len(a)
    if a[-1] == b[-1]: return lev_recursive(a[:-1], b[:-1])
    return 1 + min(lev_recursive(a[:-1], b),        # deletion
                   lev_recursive(a, b[:-1]),        # insertion
                   lev_recursive(a[:-1], b[:-1]))   # substitution

print("kitten -> sitting:", lev_recursive("kitten", "sitting"))
print("\nLength | distance | recursive calls | time (s)")
for n in range(3, 10):
    calls = 0; t0 = time.perf_counter()
    d = lev_recursive("a" * n, "b" * n)
    print(f"{n:>6} | {d:>8} | {calls:>15} | {time.perf_counter() - t0:.4f}")

**Limitation:** the recursive version recomputes the same sub-problems (`a[:i]`, `b[:j]`) over and over, so the number of calls grows **exponentially (~3^n)**, as the table shows. Only the n×m distinct sub-problems matter, which is what dynamic programming exploits (O(n·m) time).

In [ ]:
# 2. Iterative DP (ins = del = 1, sub = sub_cost)
def lev_dp(a, b, sub_cost=1):
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0] = np.arange(n + 1); D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = 0 if a[i-1] == b[j-1] else sub_cost
            D[i, j] = min(D[i-1, j] + 1,        # deletion
                          D[i, j-1] + 1,        # insertion
                          D[i-1, j-1] + sub)    # substitution / match
    return D

def show_table(a, b, D):
    return pd.DataFrame(D, index=["#"] + list(a), columns=["#"] + list(b))

# 3. DP tables for the three pairs
pairs = [("intention", "execution"), ("kitten", "sitting"), ("sunday", "saturday")]
for a, b in pairs:
    D = lev_dp(a, b)
    print(f"\n{a!r} -> {b!r}: distance = {D[-1, -1]}")
    display(show_table(a, b, D))

**Constructing a table by hand** (do this on paper for each pair and compare with the output above):

- Row 0 is `0,1,2,...,m` (building the target from nothing = m insertions); column 0 is `0,1,2,...,n` (deleting n characters).
- Every other cell is `D[i][j] = min( D[i-1][j] + 1,  D[i][j-1] + 1,  D[i-1][j-1] + (0 if a[i]==b[j] else 1) )`.
- Example for **kitten → sitting**, cell (k, s): `min(1+1, 1+1, 0+1) = 1` (substitute k→s); cell (i, i): the letters match, so `D = D[i-1][j-1] + 0 = 1`.
- The bottom-right cell is the edit distance: kitten→sitting = **3** (k→s, e→i, insert g).

In [ ]:
# 4. Backtrace
def backtrace(a, b, D, sub_cost=1):
    i, j, ops = len(a), len(b), []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1] and D[i, j] == D[i-1, j-1]:
            ops.append(("match", a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + sub_cost:
            ops.append(("substitute", a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            ops.append(("delete", a[i-1], "")); i -= 1
        else:
            ops.append(("insert", "", b[j-1])); j -= 1
    return ops[::-1]

for a, b in pairs:
    D = lev_dp(a, b); ops = backtrace(a, b, D)
    print(f"\n{a} -> {b}   (distance {D[-1, -1]})")
    for op, x, z in ops:
        if op != "match": print(f"   {op:<11} {x!r:>4} {'->' if op == 'substitute' else ''} {z!r if op != 'delete' else ''}")

# Substitution cost 2 (Jurafsky-Martin)
rows = []
for a, b in pairs:
    D1, D2 = lev_dp(a, b, 1), lev_dp(a, b, 2)
    o1, o2 = backtrace(a, b, D1, 1), backtrace(a, b, D2, 2)
    count = lambda ops, k: sum(1 for o in ops if o[0] == k)
    rows.append((f"{a} -> {b}", D1[-1, -1], D2[-1, -1], count(o1, "substitute"), count(o2, "substitute"),
                 count(o2, "insert"), count(o2, "delete")))
display(pd.DataFrame(rows, columns=["Pair", "Dist (sub=1)", "Dist (sub=2)", "#subs (sub=1)", "#subs (sub=2)", "#ins (sub=2)", "#del (sub=2)"]))

**Comparison:** with substitution cost 2, a substitution costs the same as one deletion plus one insertion, so distances increase (e.g. intention→execution goes from 5 to 8, the textbook Jurafsky–Martin value). The alignment is the same quality, but the cost reflects that substituting is "more expensive" than a single indel.

## Task 9: Spelling Correction with Edit Distance

In [ ]:
from nltk.corpus import brown
# 1. Vocabulary: lowercase alphabetic words of the Brown corpus that occur at least twice
freq = Counter(w.lower() for w in brown.words() if w.isalpha() and len(w) > 1)
freq = Counter({w: c for w, c in freq.items() if c >= 2})
dictionary = sorted(freq)
print("Dictionary size:", len(dictionary), "words")
by_len = defaultdict(list)
for w in dictionary: by_len[len(w)].append(w)

def edit_distance(a, b):                       # fast two-row Levenshtein
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

_cache = {}
def all_distances(word, max_len_diff=3):
    if word not in _cache:
        cands = (w for L in range(len(word) - max_len_diff, len(word) + max_len_diff + 1) for w in by_len.get(L, []))
        _cache[word] = [(w, edit_distance(word, w)) for w in cands]
    return _cache[word]

# 2. suggest(): version 1 = distance only (ties broken alphabetically)
def suggest(word, k=5, use_freq=False):
    word = word.lower()
    scored = all_distances(word)
    key = (lambda x: (x[1], -freq[x[0]], x[0])) if use_freq else (lambda x: (x[1], x[0]))
    return [(w, d) for w, d in sorted(scored, key=key)[:k]]

print(suggest("recieve"))
print(suggest("recieve", use_freq=True))

In [ ]:
# 3. Test on 16 misspellings
tests = {"recieve":"receive","definately":"definitely","acommodate":"accommodate","seperate":"separate",
         "occured":"occurred","untill":"until","wich":"which","tommorow":"tomorrow","beleive":"believe",
         "goverment":"government","enviroment":"environment","necesary":"necessary","begining":"beginning",
         "freind":"friend","adress":"address","knowlege":"knowledge"}
missing = [c for c in tests.values() if c not in freq]
print("Correct words missing from the dictionary (can never be found):", missing)

rows = []
for wrong, right in tests.items():
    v1 = suggest(wrong, 5)[0][0]; v2 = suggest(wrong, 5, use_freq=True)[0][0]
    rows.append((wrong, right, v1, v1 == right, v2, v2 == right))
res9 = pd.DataFrame(rows, columns=["Misspelled", "Correct", "Top-1 (distance only)", "OK v1", "Top-1 (+frequency)", "OK v2"])
display(res9)
acc1, acc2 = res9["OK v1"].mean(), res9["OK v2"].mean()
print(f"Top-1 accuracy, distance only          : {acc1:.2%}")
print(f"Top-1 accuracy, distance + frequency   : {acc2:.2%}")
top5 = np.mean([r in [w for w, _ in suggest(m, 5, use_freq=True)] for m, r in tests.items()])
print(f"Top-5 accuracy, distance + frequency   : {top5:.2%}")

**Observations:**
- With distance only, many candidates are tied at the same edit distance (for example several words at distance 1 or 2), so the winner is decided by alphabetical order, which is arbitrary and often wrong.
- Breaking ties with word frequency prefers common words ("receive" over rare lookalikes), which normally raises Top-1 accuracy; the exact numbers are printed above.
- Remaining errors come from distance ties between equally plausible words, transposition errors counted as 2 edits (`recieve`), and words missing from the dictionary.
- **Real-word errors such as "their/there" are not caught** because both words are valid dictionary entries: the misspelling has edit distance 0 to a real word, so a dictionary look-up sees nothing wrong. Detecting them needs *context* (e.g. a language model or n-gram probabilities: "over their" vs "over there").

In [ ]:
# Demonstration of the real-word problem
for w in ["their", "there", "form", "from"]:
    print(w, "in dictionary:", w in freq, "| best suggestion:", suggest(w, 1, use_freq=True))